# LeetCode #77 — Combinations: Backtracking sinh tổ hợp C(n, k)

**Mức độ:** Medium · **Chủ đề:** Backtracking

Skeleton trên LeetCode:

```python
class Solution:
    def combine(self, n: int, k: int) -> List[List[int]]:
```

Mục tiêu của notebook:

1. Hiểu **cây quyết định** của backtracking: chọn → đệ quy → bỏ chọn.
2. Tránh trùng lặp (`[1,2]` và `[2,1]`) bằng tham số `start`.
3. **Cắt tỉa (pruning)**: dừng sớm khi không còn đủ số để chọn.
4. Biết các cách khác: `itertools.combinations`, và công thức đệ quy `C(n, k) = C(n-1, k-1) + C(n-1, k)`.

# 1. Bài toán là gì?

Cho hai số nguyên `n` và `k`, trả về **tất cả** tổ hợp `k` số chọn từ `[1, n]`. Thứ tự các tổ hợp trong kết quả tùy ý.

Tổ hợp là **không có thứ tự**: `[1, 2]` và `[2, 1]` là một.

**Constraints (theo đề):**

- `1 <= n <= 20`
- `1 <= k <= n`

---

## Ví dụ

```text
n = 4, k = 2  ->  [[1,2],[1,3],[1,4],[2,3],[2,4],[3,4]]
n = 1, k = 1  ->  [[1]]
```

Số tổ hợp là `C(n, k) = n! / (k! (n-k)!)`. Với `n = 20, k = 10`: `184 756` tổ hợp.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quay lui (backtracking)
2. Hoán vị, tổ hợp, tập con và thứ tự từ điển

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [1]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [2]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [3]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [4]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [5]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.2 Hoán vị, tổ hợp, tập con và thứ tự từ điển

### Đếm

| Đối tượng | Ý nghĩa | Số lượng |
|---|---|---|
| Hoán vị của `n` phần tử | sắp **thứ tự** tất cả | `n!` |
| Chỉnh hợp chập `k` | chọn `k` phần tử **có thứ tự** | `n! / (n−k)!` |
| Tổ hợp chập `k` | chọn `k` phần tử **không** thứ tự | `C(n, k) = n! / (k! (n−k)!)` |
| Tập con | mỗi phần tử lấy/không | `2^n` |
| Hoán vị có lặp | `n` phần tử, nhóm trùng cỡ `c1, c2, …` | `n! / (c1! c2! …)` |

In [6]:
from math import factorial, comb, perm
from itertools import permutations, combinations, product

print("4! =", factorial(4), "| P(4,2) =", perm(4, 2), "| C(4,2) =", comb(4, 2), "| 2^4 =", 2 ** 4)
print("hoán vị [1,2,3]:", list(permutations([1, 2, 3])))
print("tổ hợp chập 2 của [1,2,3,4]:", list(combinations([1, 2, 3, 4], 2)))
print("hoán vị khác nhau của 'aab':", sorted(set(permutations("aab"))), "=", factorial(3) // factorial(2))
print("tích Descartes {0,1}^2:", list(product([0, 1], repeat=2)))

4! = 24 | P(4,2) = 12 | C(4,2) = 6 | 2^4 = 16
hoán vị [1,2,3]: [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
tổ hợp chập 2 của [1,2,3,4]: [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
hoán vị khác nhau của 'aab': [('a', 'a', 'b'), ('a', 'b', 'a'), ('b', 'a', 'a')] = 3
tích Descartes {0,1}^2: [(0, 0), (0, 1), (1, 0), (1, 1)]


### Thứ tự từ điển (lexicographic order)

So hai dãy như so từ trong từ điển: so phần tử **đầu tiên khác nhau**; nếu một dãy là tiền tố của dãy kia, dãy ngắn hơn đứng trước. Python so sánh list/tuple/str đúng theo kiểu này.

Liệt kê hoán vị của `[1, 2, 3]` theo thứ tự từ điển: `123, 132, 213, 231, 312, 321`.

**Quan sát về hoán vị kế tiếp:** đi từ phải sang, phần **hậu tố giảm dần** dài nhất là hậu tố "đã lớn nhất có thể". Muốn hoán vị lớn hơn kế tiếp, phải tăng phần tử **ngay trước** hậu tố đó lên một chút.

In [7]:
print([1, 2, 3] < [1, 3], "|", "abc" < "abd", "|", [1, 2] < [1, 2, 0], "|", "10" < "9")
for p in permutations([1, 2, 3]):
    i = len(p) - 1
    while i > 0 and p[i - 1] >= p[i]:
        i -= 1
    print(p, "| hậu tố giảm dần dài nhất:", p[i:])

True | True | True | True
(1, 2, 3) | hậu tố giảm dần dài nhất: (3,)
(1, 3, 2) | hậu tố giảm dần dài nhất: (3, 2)
(2, 1, 3) | hậu tố giảm dần dài nhất: (3,)
(2, 3, 1) | hậu tố giảm dần dài nhất: (3, 1)
(3, 1, 2) | hậu tố giảm dần dài nhất: (2,)
(3, 2, 1) | hậu tố giảm dần dài nhất: (3, 2, 1)


### Liên hệ với quay lui

`itertools` sinh sẵn hoán vị/tổ hợp, nhưng phỏng vấn thường yêu cầu **tự sinh** bằng quay lui (để kèm cắt tỉa, bỏ trùng, điều kiện riêng). Kích thước kết quả (`n!`, `C(n,k)`, `2^n`) cho biết độ phức tạp tối thiểu — không thuật toán nào liệt kê nhanh hơn số lời giải.

## 2.3 Áp dụng vào bài này

Liệt kê **tổ hợp** chập `k` của `1..n` (`C(n, k)` kết quả) bằng **quay lui** với `start` (chỉ chọn số lớn hơn số vừa chọn). **Cắt tỉa**: nếu số phần tử còn lại không đủ để chọn đủ `k` thì dừng.

# 3. Nền tảng

## 3.1 `math.comb` và `itertools.combinations`

Thư viện chuẩn đã có sẵn — hữu ích để **kiểm tra**, nhưng phỏng vấn sẽ yêu cầu tự viết.

In [8]:
from typing import List
import math, itertools

print(math.comb(4, 2), math.comb(20, 10))
print([list(c) for c in itertools.combinations(range(1, 5), 2)])

6 184756
[[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]


## 3.2 List là tham chiếu — phải **sao chép** khi lưu

Backtracking dùng **một** list `path` chung, thêm/bớt liên tục. Nếu `result.append(path)` thì mọi phần tử của `result` là **cùng một** list, và cuối cùng đều rỗng.

In [9]:
result = []
path = []
for x in [1, 2, 3]:
    path.append(x)
    result.append(path)        # lưu tham chiếu
    path.pop()
print("Không sao chép:", result)

result = []
for x in [1, 2, 3]:
    path.append(x)
    result.append(path[:])     # lưu bản sao
    path.pop()
print("Có sao chép  :", result)

Không sao chép: [[], [], []]
Có sao chép  : [[1], [2], [3]]


# 4. Ý tưởng chính: cây quyết định

Xây tổ hợp theo thứ tự **tăng dần**: sau khi chọn số `i`, chỉ chọn tiếp các số `> i`. Nhờ vậy mỗi tổ hợp chỉ sinh ra **một lần** (dạng đã sắp xếp).

```text
                         []
         /            |         \        \
       [1]           [2]        [3]      [4]
     /  |  \         /  \        |
 [1,2][1,3][1,4] [2,3][2,4]    [3,4]    (k = 2 -> lá)
```

```text
backtrack(start, path):
    if len(path) == k: lưu path[:]; return
    for i in range(start, n + 1):
        path.append(i)          # chọn
        backtrack(i + 1, path)  # đi sâu
        path.pop()              # bỏ chọn (quay lui)
```

In [10]:
def combine_basic(n: int, k: int) -> List[List[int]]:
    result = []
    path = []
    def backtrack(start):
        if len(path) == k:
            result.append(path[:])
            return
        for i in range(start, n + 1):
            path.append(i)
            backtrack(i + 1)
            path.pop()
    backtrack(1)
    return result

print(combine_basic(4, 2))

[[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]


# 5. Mô phỏng

In [11]:
def trace_combine(n: int, k: int) -> List[List[int]]:
    result, path = [], []
    def backtrack(start, depth):
        indent = "  " * depth
        if len(path) == k:
            print(f"{indent}lưu {path}")
            result.append(path[:])
            return
        for i in range(start, n + 1):
            path.append(i)
            print(f"{indent}chọn {i} -> {path}")
            backtrack(i + 1, depth + 1)
            path.pop()
    backtrack(1, 0)
    return result

_ = trace_combine(4, 2)

chọn 1 -> [1]
  chọn 2 -> [1, 2]
    lưu [1, 2]
  chọn 3 -> [1, 3]
    lưu [1, 3]
  chọn 4 -> [1, 4]
    lưu [1, 4]
chọn 2 -> [2]
  chọn 3 -> [2, 3]
    lưu [2, 3]
  chọn 4 -> [2, 4]
    lưu [2, 4]
chọn 3 -> [3]
  chọn 4 -> [3, 4]
    lưu [3, 4]
chọn 4 -> [4]


Nhận xét: ở nhánh `chọn 4 -> [4]`, không còn số nào lớn hơn 4 để chọn → nhánh này **vô ích**. Với `k` lớn, những nhánh vô ích như vậy rất nhiều.

# 6. Cắt tỉa (pruning)

Đang có `len(path)` số, cần thêm `need = k - len(path)` số. Nếu chọn `i`, còn lại các số `i+1 .. n` (có `n - i` số) phải đủ cho `need - 1` số nữa:

```text
n - i >= need - 1   ⇔   i <= n - need + 1
```

Vậy vòng lặp chỉ cần chạy tới `n - (k - len(path)) + 1`.

In [12]:
def combine_pruned(n: int, k: int) -> List[List[int]]:
    result, path = [], []
    def backtrack(start):
        if len(path) == k:
            result.append(path[:])
            return
        need = k - len(path)
        for i in range(start, n - need + 2):
            path.append(i)
            backtrack(i + 1)
            path.pop()
    backtrack(1)
    return result


def count_calls(prune, n, k):
    calls = 0
    result, path = [], []
    def backtrack(start):
        nonlocal calls
        calls += 1
        if len(path) == k:
            result.append(path[:])
            return
        end = n - (k - len(path)) + 1 if prune else n
        for i in range(start, end + 1):
            path.append(i)
            backtrack(i + 1)
            path.pop()
    backtrack(1)
    return calls

for n, k in [(4, 2), (10, 8), (20, 18)]:
    print(f"n={n:>2}, k={k:>2}: số lời gọi không cắt tỉa = {count_calls(False, n, k):>8}, có cắt tỉa = {count_calls(True, n, k):>6}, C(n,k) = {math.comb(n, k)}")

n= 4, k= 2: số lời gọi không cắt tỉa =       11, có cắt tỉa =     10, C(n,k) = 6
n=10, k= 8: số lời gọi không cắt tỉa =     1013, có cắt tỉa =    165, C(n,k) = 45


n=20, k=18: số lời gọi không cắt tỉa =  1048555, có cắt tỉa =   1330, C(n,k) = 190


Với `k` gần `n`, cắt tỉa giảm số lời gọi rất nhiều.

# 7. Cách khác: công thức Pascal

Mỗi tổ hợp `k` số từ `[1, n]` hoặc **chứa** `n` hoặc **không**:

```text
C(n, k) = { c + [n] : c ∈ C(n-1, k-1) }  ∪  C(n-1, k)
```

Viết đệ quy trực tiếp theo công thức này:

In [13]:
def combine_pascal(n: int, k: int) -> List[List[int]]:
    if k == 0:
        return [[]]
    if n < k:
        return []
    with_n = [c + [n] for c in combine_pascal(n - 1, k - 1)]
    without_n = combine_pascal(n - 1, k)
    return without_n + with_n

print(combine_pascal(4, 2))

[[1, 2], [1, 3], [2, 3], [1, 4], [2, 4], [3, 4]]


# 8. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def combine(self, n: int, k: int) -> List[List[int]]:
        result, path = [], []

        def backtrack(start: int) -> None:
            if len(path) == k:
                result.append(path[:])
                return
            need = k - len(path)
            for i in range(start, n - need + 2):
                path.append(i)
                backtrack(i + 1)
                path.pop()

        backtrack(1)
        return result

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual if len(str(actual)) < 70 else str(actual)[:67] + '...'}")


def norm(res):
    return sorted(sorted(c) for c in res)


sol = Solution()
check("Ví dụ 1", norm(sol.combine(4, 2)), [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]])
check("Ví dụ 2", sol.combine(1, 1), [[1]])
check("k = n", sol.combine(3, 3), [[1, 2, 3]])
check("k = 1", norm(sol.combine(3, 1)), [[1], [2], [3]])

for n in range(1, 11):
    for k in range(1, n + 1):
        expected = [list(c) for c in itertools.combinations(range(1, n + 1), k)]
        assert norm(sol.combine(n, k)) == expected == norm(combine_basic(n, k)) == norm(combine_pascal(n, k))
print("Mọi 1 <= k <= n <= 10: khớp itertools.combinations")
print("n = 20, k = 10:", len(sol.combine(20, 10)), "tổ hợp, đúng bằng", math.comb(20, 10))

Ví dụ 1            PASS  [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
Ví dụ 2            PASS  [[1]]
k = n              PASS  [[1, 2, 3]]
k = 1              PASS  [[1], [2], [3]]
Mọi 1 <= k <= n <= 10: khớp itertools.combinations


n = 20, k = 10: 184756 tổ hợp, đúng bằng 184756


# 9. Vì sao đúng? Độ phức tạp?

- **Không trùng:** mỗi `path` sinh ra là dãy **tăng dần nghiêm ngặt** (luôn chọn `i ≥ start` với `start = số trước + 1`), và mỗi tập `k` số có đúng một cách viết tăng dần.
- **Không thiếu:** với tập bất kỳ `{a1 < a2 < … < ak}`, đường đi chọn `a1`, rồi `a2`, … tồn tại trong cây (cắt tỉa chỉ bỏ các nhánh không thể đủ `k` số).

```text
Time:  O(k · C(n, k))  — có C(n, k) kết quả, mỗi lần sao chép tốn O(k)
Space: O(k) cho đệ quy và path (không tính output)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Lưu `path` thay vì `path[:]`

In [16]:
def wrong_no_copy(n: int, k: int) -> List[List[int]]:
    result, path = [], []
    def backtrack(start):
        if len(path) == k:
            result.append(path)                    # SAI
            return
        for i in range(start, n + 1):
            path.append(i)
            backtrack(i + 1)
            path.pop()
    backtrack(1)
    return result

print("Sai :", wrong_no_copy(4, 2))

Sai : [[], [], [], [], [], []]


## Lỗi 2: Bắt đầu lại từ 1 thay vì `i + 1`

Sinh ra **chỉnh hợp** (có thứ tự, và có cả số lặp), không phải tổ hợp.

In [17]:
def wrong_restart(n: int, k: int) -> List[List[int]]:
    result, path = [], []
    def backtrack(start):
        if len(path) == k:
            result.append(path[:])
            return
        for i in range(start, n + 1):
            path.append(i)
            backtrack(1)                           # SAI: phải là i + 1
            path.pop()
    backtrack(1)
    return result

print("Sai :", len(wrong_restart(4, 2)), "kết quả, ví dụ", wrong_restart(4, 2)[:5], "| Đúng:", math.comb(4, 2), "kết quả")

Sai : 16 kết quả, ví dụ [[1, 1], [1, 2], [1, 3], [1, 4], [2, 1]] | Đúng: 6 kết quả


## Lỗi 3: Dùng `start + 1` thay vì `i + 1`

In [18]:
def wrong_start_plus(n: int, k: int) -> List[List[int]]:
    result, path = [], []
    def backtrack(start):
        if len(path) == k:
            result.append(path[:])
            return
        for i in range(start, n + 1):
            path.append(i)
            backtrack(start + 1)                   # SAI
            path.pop()
    backtrack(1)
    return result

print("Sai :", wrong_start_plus(4, 2))

Sai : [[1, 2], [1, 3], [1, 4], [2, 2], [2, 3], [2, 4], [3, 2], [3, 3], [3, 4], [4, 2], [4, 3], [4, 4]]


`[2, 2]`, `[3, 2]`... xuất hiện vì lần chọn sau không phụ thuộc số vừa chọn.

## Lỗi 4: Sai biên cắt tỉa

Biên đúng là `range(start, n - need + 2)` (vì `range` loại trừ đầu mút). Viết `n - need + 1` sẽ **bỏ sót** tổ hợp chứa số lớn nhất có thể.

# 11. Mẹo nhận diện pattern

**Backtracking** khi đề yêu cầu **liệt kê tất cả** (tổ hợp, hoán vị, tập con, phân hoạch...). Phân biệt:

| Bài | Đặc điểm | Đệ quy tiếp từ |
|---|---|---|
| #77 Combinations | chọn đúng `k`, không thứ tự | `i + 1` |
| #78 Subsets | mọi kích thước | `i + 1`, lưu ở **mọi** nút |
| #39 Combination Sum | được dùng lại | `i` |
| #40 Combination Sum II | có số trùng trong input | `i + 1`, bỏ `nums[i] == nums[i-1]` cùng tầng |
| #46 Permutations | có thứ tự | mọi số chưa dùng (`used[]`) |

## Template

```python
def backtrack(start):
    if đủ điều kiện: result.append(path[:]); return
    for i in range(start, end):
        path.append(choice[i])
        backtrack(i + 1)
        path.pop()
```

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Vẽ cây quyết định cho `n = 5, k = 3` **có cắt tỉa**. Có bao nhiêu lá?

## Bài 2 — Điền chỗ trống

In [19]:
def exercise_2_combine(n: int, k: int) -> List[List[int]]:
    result, path = [], []
    def backtrack(start):
        # TODO: điều kiện dừng, vòng lặp chọn -> đệ quy -> bỏ chọn
        pass
    backtrack(1)
    return result


# assert sorted(exercise_2_combine(4, 2)) == [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
# assert exercise_2_combine(1, 1) == [[1]]
# assert len(exercise_2_combine(10, 4)) == 210

## Bài 3 — Biến thể: Combination Sum III (LeetCode #216)

Tìm mọi tổ hợp `k` số **phân biệt** trong `1..9` có tổng bằng `n`.

```python
combination_sum3(3, 7) -> [[1, 2, 4]]
combination_sum3(3, 9) -> [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
combination_sum3(4, 1) -> []
```

In [20]:
def exercise_3_combination_sum3(k: int, n: int) -> List[List[int]]:
    # TODO
    pass


# assert sorted(exercise_3_combination_sum3(3, 7)) == [[1, 2, 4]]
# assert sorted(exercise_3_combination_sum3(3, 9)) == [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
# assert exercise_3_combination_sum3(4, 1) == []

## Bài 4 — Biến thể: sinh tổ hợp không đệ quy (thứ tự từ điển)

Sinh **tổ hợp kế tiếp** theo thứ tự từ điển: từ phải sang trái, tìm vị trí `j` đầu tiên có thể tăng (`c[j] < n - k + 1 + j` với chỉ số 0), tăng nó lên 1, rồi đặt các phần tử sau thành dãy liên tiếp.

```python
next_combination([1, 2, 4], 4) -> [1, 3, 4]
next_combination([2, 3, 4], 4) -> None     # tổ hợp cuối
```

In [21]:
def exercise_4_next_combination(c: List[int], n: int):
    # TODO: trả list mới, hoặc None
    pass


# assert exercise_4_next_combination([1, 2, 4], 4) == [1, 3, 4]
# assert exercise_4_next_combination([1, 2], 4) == [1, 3]
# assert exercise_4_next_combination([2, 3, 4], 4) is None

## Bài 5 — Mini challenge: tổ hợp thứ `m` (không sinh hết)

Trả về tổ hợp thứ `m` (đếm từ 0) trong thứ tự từ điển của mọi tổ hợp `k` số từ `[1, n]`, dùng `math.comb`, không sinh ra tất cả.

```python
kth_combination(4, 2, 0) -> [1, 2]
kth_combination(4, 2, 3) -> [2, 3]
kth_combination(4, 2, 5) -> [3, 4]
```

In [22]:
def exercise_5_kth_combination(n: int, k: int, m: int) -> List[int]:
    # TODO
    pass


# assert exercise_5_kth_combination(4, 2, 0) == [1, 2]
# assert exercise_5_kth_combination(4, 2, 3) == [2, 3]
# assert exercise_5_kth_combination(4, 2, 5) == [3, 4]

# 13. Gợi ý

**Bài 1**
- Mức 1: số lá = `C(5, 3) = 10`; cắt tỉa đảm bảo không có nhánh "chết".

**Bài 3**
- Mức 1: backtracking trên `1..9`, thêm tham số `remain = n - sum(path)`.
- Mức 2: dừng sớm khi `i > remain` (các số sau còn lớn hơn).

**Bài 4**
- Mức 1: phần tử ở chỉ số `j` (0-based) có giá trị tối đa `n - k + 1 + j`.
- Mức 2: sau khi tăng `c[j]`, đặt `c[t] = c[j] + (t - j)` cho `t > j`.

**Bài 5**
- Mức 1: thử số nhỏ nhất `x` cho vị trí đầu: có `comb(n - x, k - 1)` tổ hợp bắt đầu bằng `x`.
- Mức 2: nếu `m < comb(n - x, k - 1)` thì chọn `x`, giảm `k`; ngược lại `m -= comb(...)` và thử `x + 1`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
_ = trace_combine(5, 3)
print("Số lá:", math.comb(5, 3))

chọn 1 -> [1]
  chọn 2 -> [1, 2]
    chọn 3 -> [1, 2, 3]
      lưu [1, 2, 3]
    chọn 4 -> [1, 2, 4]
      lưu [1, 2, 4]
    chọn 5 -> [1, 2, 5]
      lưu [1, 2, 5]
  chọn 3 -> [1, 3]
    chọn 4 -> [1, 3, 4]
      lưu [1, 3, 4]
    chọn 5 -> [1, 3, 5]
      lưu [1, 3, 5]
  chọn 4 -> [1, 4]
    chọn 5 -> [1, 4, 5]
      lưu [1, 4, 5]
  chọn 5 -> [1, 5]
chọn 2 -> [2]
  chọn 3 -> [2, 3]
    chọn 4 -> [2, 3, 4]
      lưu [2, 3, 4]
    chọn 5 -> [2, 3, 5]
      lưu [2, 3, 5]
  chọn 4 -> [2, 4]
    chọn 5 -> [2, 4, 5]
      lưu [2, 4, 5]
  chọn 5 -> [2, 5]
chọn 3 -> [3]
  chọn 4 -> [3, 4]
    chọn 5 -> [3, 4, 5]
      lưu [3, 4, 5]
  chọn 5 -> [3, 5]
chọn 4 -> [4]
  chọn 5 -> [4, 5]
chọn 5 -> [5]
Số lá: 10


(Hàm `trace_combine` không cắt tỉa nên còn in các nhánh `chọn 4`, `chọn 5` không dẫn tới lá; bản cắt tỉa sẽ không đi vào những nhánh đó.)

## Đáp án Bài 2

In [24]:
def answer_2_combine(n: int, k: int) -> List[List[int]]:
    return combine_pruned(n, k)

assert sorted(answer_2_combine(4, 2)) == [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
assert answer_2_combine(1, 1) == [[1]]
assert len(answer_2_combine(10, 4)) == 210
print("Bài 2 PASS (giống combine_pruned ở mục 6)")

Bài 2 PASS (giống combine_pruned ở mục 6)


## Đáp án Bài 3

In [25]:
def answer_3_combination_sum3(k: int, n: int) -> List[List[int]]:
    result, path = [], []
    def backtrack(start, remain):
        if len(path) == k:
            if remain == 0:
                result.append(path[:])
            return
        for i in range(start, 10):
            if i > remain:
                break
            path.append(i)
            backtrack(i + 1, remain - i)
            path.pop()
    backtrack(1, n)
    return result

assert sorted(answer_3_combination_sum3(3, 7)) == [[1, 2, 4]]
assert sorted(answer_3_combination_sum3(3, 9)) == [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
assert answer_3_combination_sum3(4, 1) == []
for k in range(1, 10):
    for n in range(1, 46):
        expected = [list(c) for c in itertools.combinations(range(1, 10), k) if sum(c) == n]
        assert sorted(answer_3_combination_sum3(k, n)) == expected
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
def answer_4_next_combination(c: List[int], n: int):
    k = len(c)
    c = c[:]
    for j in range(k - 1, -1, -1):
        if c[j] < n - k + 1 + j:
            c[j] += 1
            for t in range(j + 1, k):
                c[t] = c[j] + (t - j)
            return c
    return None

assert answer_4_next_combination([1, 2, 4], 4) == [1, 3, 4]
assert answer_4_next_combination([1, 2], 4) == [1, 3]
assert answer_4_next_combination([2, 3, 4], 4) is None
for n in range(1, 8):
    for k in range(1, n + 1):
        c, seq = list(range(1, k + 1)), []
        while c is not None:
            seq.append(c)
            c = answer_4_next_combination(c, n)
        assert seq == [list(x) for x in itertools.combinations(range(1, n + 1), k)]
print("Bài 4 PASS (sinh đúng toàn bộ dãy tổ hợp theo thứ tự từ điển)")

Bài 4 PASS (sinh đúng toàn bộ dãy tổ hợp theo thứ tự từ điển)


## Đáp án Bài 5

In [27]:
def answer_5_kth_combination(n: int, k: int, m: int) -> List[int]:
    result = []
    x = 1
    while k > 0:
        cnt = math.comb(n - x, k - 1)       # số tổ hợp bắt đầu bằng x
        if m < cnt:
            result.append(x)
            k -= 1
        else:
            m -= cnt
        x += 1
    return result

assert answer_5_kth_combination(4, 2, 0) == [1, 2]
assert answer_5_kth_combination(4, 2, 3) == [2, 3]
assert answer_5_kth_combination(4, 2, 5) == [3, 4]
for n in range(1, 8):
    for k in range(1, n + 1):
        all_c = [list(c) for c in itertools.combinations(range(1, n + 1), k)]
        assert [answer_5_kth_combination(n, k, m) for m in range(len(all_c))] == all_c
print("Bài 5 PASS")
print("Tổ hợp thứ 100000 của C(20, 10):", answer_5_kth_combination(20, 10, 100000))

Bài 5 PASS
Tổ hợp thứ 100000 của C(20, 10): [2, 3, 4, 6, 10, 11, 13, 15, 17, 19]


# 15. Tổng kết

> Backtracking: **chọn → đệ quy → bỏ chọn**. Đệ quy tiếp từ `i + 1` để mỗi tổ hợp chỉ sinh một lần (dạng tăng dần). Lưu `path[:]`. Cắt tỉa: `i <= n - (k - len(path)) + 1`.

---

## Checklist trước khi nộp

- [ ] Lưu **bản sao** `path[:]`?
- [ ] Đệ quy với `i + 1` (không phải `start + 1` hay `1`)?
- [ ] `path.pop()` sau lời gọi đệ quy?
- [ ] Biên cắt tỉa đúng (`range(start, n - need + 2)`)?